# SegFormer Vitiligo Finetuning

This notebook finetunes the vitiligo segmentation model on Google Colab using CUDA GPU.

It starts from the **port wine stain 4-class checkpoint** (`segformer_b2_4class_port_wine_stain_finetune/best`),
which already has the 4-class architecture (background, vitiligo, melasma, port_wine_stain).
Training on vitiligo data improves class 1 behaviour without breaking the other classes.

Before running this notebook, create the upload package locally from the `Makeup` project root:

```powershell
Compress-Archive -Path @(
  "backend-skin-analyzer\training\SegFormer\vitiligo\train_vitiligo.py",
  "virtiligo\dataset\processed",
  "backend-skin-analyzer\training\checkpoints\SegFormer\segformer_b2_4class_port_wine_stain_finetune\best",
  "backend-skin-analyzer\requirements.txt"
) -DestinationPath "vitiligo_training_pack.zip"
```

Then upload `vitiligo_training_pack.zip` in this notebook.

## 1. Enable GPU

In Colab, go to:

`Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU -> Save`

In [16]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


## 2. Upload Training Package

Upload `vitiligo_training_pack.zip` when prompted.

In [17]:
from google.colab import files

uploaded = files.upload()

Saving vitiligo_training_pack_v2.zip to vitiligo_training_pack_v2.zip


## 3. Unzip Project Files

In [18]:
from pathlib import Path

zip_files = sorted(Path('/content').glob('vitiligo_training_pack*.zip'))
print('found zip files:', [str(p) for p in zip_files])
assert zip_files, 'Upload vitiligo_training_pack.zip first, then rerun this cell.'
zip_path = zip_files[-1]

!rm -rf /content/Back_Lumiere
!unzip -q "{zip_path}" -d /content/Back_Lumiere
%cd /content/Back_Lumiere
!find virtiligo/dataset/processed -maxdepth 2 -type f | wc -l

found zip files: ['/content/vitiligo_training_pack.zip']
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
/content/Back_Lumiere
find: ‘virtiligo/dataset/processed’: No such file or directory
0


## 4. Install Training Dependencies

In [19]:
!pip install -q transformers safetensors tqdm
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121


## 5. Verify Dataset

In [20]:
!find /content/Back_Lumiere -maxdepth 4 -type d


/content/Back_Lumiere
/content/Back_Lumiere/processed
/content/Back_Lumiere/processed/splits
/content/Back_Lumiere/processed/images
/content/Back_Lumiere/processed/masks
/content/Back_Lumiere/best


In [21]:
from pathlib import Path
from PIL import Image
import numpy as np

dataset_dir = Path("processed")
print("images:", len(list((dataset_dir / "images").iterdir())))
print("masks:", len(list((dataset_dir / "masks").glob("*.png"))))
print("train:", len((dataset_dir / "splits" / "train.txt").read_text().splitlines()))
print("val:", len((dataset_dir / "splits" / "val.txt").read_text().splitlines()))
print("test:", len((dataset_dir / "splits" / "test.txt").read_text().splitlines()))

# Verify mask values — vitiligo pixels must be 1
sample_name = (dataset_dir / "splits" / "train.txt").read_text().splitlines()[0]
mask = np.array(Image.open(dataset_dir / "masks" / f"{sample_name}.png"))
print("sample mask:", sample_name, "unique values:", np.unique(mask).tolist())
print("expected: [0, 1] — 0=background, 1=vitiligo")

images: 100
masks: 100
train: 70
val: 15
test: 15
sample mask: 1 unique values: [0, 1]
expected: [0, 1] — 0=background, 1=vitiligo


## 6. Verify Starting Checkpoint

Confirm the port wine stain 4-class checkpoint loaded correctly.

In [22]:
import json
from pathlib import Path

checkpoint = Path("best")
config = json.loads((checkpoint / "config.json").read_text())
print("num_labels:", config.get("num_labels"))
print("id2label:", config.get("id2label"))
print("Expected: 4 classes — 0=background, 1=vitiligo, 2=melasma, 3=port_wine_stain")

num_labels: None
id2label: {'0': 'background', '1': 'vitiligo', '2': 'melasma_like_hyperpigmentation', '3': 'port_wine_stain'}
Expected: 4 classes — 0=background, 1=vitiligo, 2=melasma, 3=port_wine_stain


In [23]:
import torch
print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("cuda version:", torch.version.cuda)


torch version: 2.11.0+cu128
cuda available: True
cuda version: 12.8


## 7. Smoke Test: Train 1 Epoch

Run one epoch first to confirm paths, GPU, checkpoint loading, and saving all work.

In [24]:
!python train_vitiligo.py \
  --dataset-dir processed \
  --checkpoint best \
  --output-dir vitiligo_finetune_smoke_test \
  --image-size 512 \
  --batch-size 4 \
  --epochs 1 \
  --learning-rate 5e-5 \
  --device cuda

using_device=cuda
Loading weights: 100% 380/380 [00:00<00:00, 732.59it/s, Materializing param=segformer.encoder.patch_embeddings.3.proj.weight]
epoch=001 train_loss=1.7392 val_loss=1.6175 val_vitiligo_iou=0.0000
Writing model shards: 100% 1/1 [00:00<00:00,  2.64it/s]
Writing model shards: 100% 1/1 [00:00<00:00,  4.09it/s]
best_val_vitiligo_iou=0.0000
saved_best=vitiligo_finetune_smoke_test/best
saved_last=vitiligo_finetune_smoke_test/last


## 8. Full Training

If the smoke test succeeds, run full training.
If Colab reports CUDA out of memory, change `--batch-size 4` to `--batch-size 2`.

In [25]:
!python train_vitiligo.py \
  --dataset-dir processed \
  --checkpoint best \
  --output-dir vitiligo_finetune \
  --image-size 512 \
  --batch-size 4 \
  --epochs 50 \
  --learning-rate 5e-5 \
  --device cuda


using_device=cuda
Loading weights: 100% 380/380 [00:00<00:00, 762.71it/s, Materializing param=segformer.encoder.patch_embeddings.3.proj.weight]
epoch=001 train_loss=1.7576 val_loss=1.5206 val_vitiligo_iou=0.0000
Writing model shards: 100% 1/1 [00:00<00:00,  3.72it/s]
epoch=002 train_loss=1.4370 val_loss=1.4313 val_vitiligo_iou=0.0000
epoch=003 train_loss=1.2041 val_loss=1.2521 val_vitiligo_iou=0.0000
epoch=004 train_loss=1.1390 val_loss=1.1861 val_vitiligo_iou=0.0000
epoch=005 train_loss=1.0663 val_loss=1.1134 val_vitiligo_iou=0.0000
Writing model shards: 100% 1/1 [00:00<00:00,  4.97it/s]
epoch=006 train_loss=1.0483 val_loss=1.0692 val_vitiligo_iou=0.0000
Writing model shards: 100% 1/1 [00:00<00:00,  3.66it/s]
epoch=007 train_loss=0.9858 val_loss=1.0143 val_vitiligo_iou=0.0007
Writing model shards: 100% 1/1 [00:00<00:00,  4.29it/s]
epoch=008 train_loss=0.9160 val_loss=0.9613 val_vitiligo_iou=0.0008
Writing model shards: 100% 1/1 [00:00<00:00,  4.89it/s]
epoch=009 train_loss=0.8775 val_

## 9. Inspect Training History

In [26]:
import json
from pathlib import Path
history_path = Path("vitiligo_finetune/training_history.json")
history = json.loads(history_path.read_text())
print("epochs:", len(history))
print("last:", history[-1])
print("best:", max(history, key=lambda row: row["val_vitiligo_iou"]))

epochs: 50
last: {'epoch': 50, 'train_loss': 0.6596549517578549, 'val_loss': 0.7075607776641846, 'val_vitiligo_iou': 0.4456340393822834}
best: {'epoch': 22, 'train_loss': 0.7005004816585116, 'val_loss': 0.7228428572416306, 'val_vitiligo_iou': 0.49330217319333486}


## 10. Zip Result Checkpoint

In [27]:
!zip -r segformer_b2_4class_vitiligo_finetune.zip vitiligo_finetune


  adding: vitiligo_finetune/ (stored 0%)
  adding: vitiligo_finetune/training_history.json (deflated 75%)
  adding: vitiligo_finetune/label_mapping.json (deflated 46%)
  adding: vitiligo_finetune/last/ (stored 0%)
  adding: vitiligo_finetune/last/model.safetensors (deflated 7%)
  adding: vitiligo_finetune/last/config.json (deflated 58%)
  adding: vitiligo_finetune/best/ (stored 0%)
  adding: vitiligo_finetune/best/model.safetensors (deflated 7%)
  adding: vitiligo_finetune/best/config.json (deflated 58%)


## 11. Download Result

In [28]:
from google.colab import files

files.download("segformer_b2_4class_vitiligo_finetune.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>